# Análise de Cobertura de Saúde Bucal no Brasil

## Contexto e Problema:
Esse projeto nasce da necessidade de entender as lacunas no atendimento odontológico público no Brasil, servindo como base estratégica para a ONG **Turma do Bem**. O principal desafio é identificar regiões onde a cobertura de saúde bucal na atenção primária é insuficiente para atender a demanda da população, especialmente em estados com alta densidade demográfica.

## Objetivo:
Realizar uma Análise Exploratória de Dados (EDA) para identificar padrões, tendências e disparidades regionais na cobertura de saúde bucal entre 2012 e 2021. Esta análise servirá de base para a criação de um modelo preditivo que ajude a antecipar carências assistenciais e otimizar a alocação de dentistas voluntários.

## Relação com o Projeto:
A base de dados selecionada permite visualizar o impacto social do projeto, conectando a análise de dados com a entrega de uma solução web responsiva que facilitará a visualização dessas carências em tempo real.

### Importando dados

In [ ]:
#importando bibliotecas
import pandas as pd
import plotly.express as px

In [ ]:
#carregando arquivos
df = pd.read_csv("1.2.csv", sep=';', decimal=',')

In [ ]:
print(df.describe())

               Ano       Valor
count   280.000000  280.000000
mean   2016.500000    0.603684
std       2.877424    0.174318
min    2012.000000    0.214800
25%    2014.000000    0.477200
50%    2016.500000    0.601450
75%    2019.000000    0.707575
max    2021.000000    0.975400


In [ ]:
#filtrando dados
df_brasil = df[df['Granularidade Geográfica'] == 'Brasil'].copy()
df_estados = df[df['Granularidade Geográfica'] != 'Brasil'].copy()
df_2021 = df_estados[df_estados['Ano'] == 2021].copy()

In [ ]:
#criando o dicionário de regiões para enriquecer a análise
region_map = {
    'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
}
df['Regiao'] = df['UF'].map(region_map)



---



### Padrões ou Tendências Temporais
**O que faz:** Mostra como a cobertura mudou no Brasil ao longo dos anos.

**Explicação:** Este gráfico de linha permite identificar se a saúde bucal no país está progredindo, estagnada ou retrocedendo. É a base para justificar se as políticas públicas atuais são suficientes.

**Análise do Projeto:** Ao monitorar a linha do tempo nacional, observamos se o avanço é real ou apenas nominal. Em estados com populações gigantescas (SP e RJ), qualquer estagnação na linha temporal é crítica. Isso justifica o trabalho voluntário da **Turma do Bem**, que atua como uma rede de segurança onde o crescimento do sistema público não acompanha a demanda demográfica.



In [ ]:
#filtramos apenas a linha que representa o consolidado do Brasil
df_brasil = df[df['Granularidade Geográfica'] == 'Brasil']

fig1 = px.line(df_brasil, x='Ano', y='Valor',
              title='Tendência Temporal: Evolução da Cobertura de Saúde Bucal no Brasil',
              labels={'Valor': 'Cobertura (%)', 'Ano': 'Ano'},
              markers=True)
fig1.show()

### Correlações entre Variáveis
**O que faz:** Cruza dois dados diferentes para ver se um influencia o outro.

**Explicação:** Como seu dataset foca em cobertura, vamos correlacionar o passado (2012) com o presente (2021). Se os pontos formarem uma linha subindo, significa que estados que investiram cedo continuam colhendo frutos.

**Análise do Projeto:** A dispersão revela que estados populosos que começaram com baixa cobertura raramente conseguem dar saltos drásticos sozinhos. Essa correlação de "baixa cobertura persistente" em SP e RJ prova que o sistema público está saturado, tornando a intervenção de dentistas voluntários essencial para quebrar esse ciclo de desassistência.

In [ ]:
#criando um dataframe comparativo entre 2012 e 2021
df_estados = df[df['Granularidade Geográfica'] != 'Brasil']
v12 = df_estados[df_estados['Ano'] == 2012][['UF', 'Valor']].rename(columns={'Valor': 'v12'})
v21 = df_estados[df_estados['Ano'] == 2021][['UF', 'Valor']].rename(columns={'Valor': 'v21'})
df_corr = pd.merge(v12, v21, on='UF')

fig2 = px.scatter(df_corr, x='v12', y='v21', text='UF',
                 title='Correlação: Cobertura de Saúde Bucal (2012 vs 2021)',
                 labels={'v12': 'Cobertura em 2012 (%)', 'v21': 'Cobertura em 2021 (%)'},
                 trendline="ols") #adiciona a linha de tendência
fig2.show()

### Distribuições Relevantes
**O que faz:** Mostra a frequência dos valores.

**Explicação:** O histograma agrupa os estados por "faixas de cobertura". Ele responde: "A maioria dos estados brasileiros está em qual nível de atendimento?". Isso ajuda a entender a saúde bucal como um todo no país.

**Análise do Projeto:** Se a distribuição mostra que grandes centros urbanos estão nas faixas de menor cobertura, temos a prova estatística da falha de alcance. Onde a barra do histograma é mais alta em valores baixos de cobertura, é onde a **Turma do Bem** encontra sua maior razão de existir, transformando o voluntariado em política ativa de saúde.

In [ ]:
#usando apenas os dados mais recentes (2021)
df_2021 = df_estados[df_estados['Ano'] == 2021]

fig3 = px.histogram(df_2021, x='Valor', nbins=10,
                   title='Distribuição: Frequência de Cobertura entre os Estados (2021)',
                   labels={'Valor': 'Faixa de Cobertura (%)'})
fig3.show()

### Estruturas, Grupos ou Outliers
**O que faz:** Identifica comportamentos atípicos.

**Explicação:** O Boxplot separa os dados em quartis. Qualquer ponto que aparecer muito longe da "caixa" central é um Outlier. No seu caso, estados com cobertura baixíssima (como DF) ou altíssima (como Piauí) se destacam como grupos que precisam de estudos específicos.

**Análise do Projeto:** Quando estados populosos aparecem como outliers (pontos fora da curva) com cobertura inferior à média nacional, fica evidente que o sistema público não atende a todos devido à escala populacional. O trabalho voluntário foca justamente nesses "vazios" onde os números oficiais mostram que a estrutura falhou.

In [ ]:
fig4 = px.box(df_estados, x='Ano', y='Valor',
             title='Estruturas e Outliers: Dispersão da Cobertura por Ano',
             points="all", #mostra todos os estados ao lado da caixa
             labels={'Valor': 'Cobertura (%)'})
fig4.show()

### Comparações entre Grupos
**O que faz:** Coloca os grupos lado a lado para ver quem é maior.

**Explicação:** Vamos comparar todos os estados em 2021. Este gráfico é o "mapa da fome" da saúde bucal: ele mostra visualmente onde a Turma do Bem deve agir com urgência (estados com as barras menores).

**Análise do Projeto:** Comparar as barras de SP e RJ com estados de menor população revela o abismo da assistência. Mesmo com mais recursos, a baixa cobertura percentual nesses estados populosos valida o impacto social da **Turma do Bem**: sem os dentistas voluntários, a lacuna entre o que o estado oferece e o que a população precisa seria ainda mais profunda.

In [ ]:
fig5 = px.bar(df_2021.sort_values('Valor', ascending=False),
             x='UF', y='Valor', color='Valor',
             title='Comparação: Ranking de Cobertura por Estado (2021)',
             labels={'Valor': 'Cobertura (%)', 'UF': 'Estado'})
fig5.show()

# Resultados da Análise:
Através dos 5 gráficos gerados, observamos que, embora exista uma tendência de estabilidade ou leve crescimento na cobertura nacional, a desigualdade entre os estados é gritante. Estados populosos como São Paulo e Rio de Janeiro apresentam taxas de cobertura proporcionalmente baixas, aparecendo muitas vezes como "outliers" ou grupos de baixa performance em comparação ao Nordeste, por exemplo.

## Justificativa da Solução:
Esses dados provam estatisticamente que o sistema público não consegue escalar o atendimento na mesma velocidade que a população cresce nos grandes centros. Isso valida totalmente a proposta da **Turma do Bem**: o trabalho voluntário não é apenas um complemento, mas uma necessidade vital para preencher os "vazios" deixados pelo Estado.

## Próxima Sprint (Machine Learning):
Com a base limpa e os padrões identificados, o próximo passo será treinar um modelo de ML usando a coluna Valor como variável alvo (Target). O objetivo será prever quedas na cobertura ou identificar estados que entrarão em zona crítica, permitindo que a ONG se antecipe e mobilize voluntários antes que o problema se agrave.